<a href="https://colab.research.google.com/github/ankita-rath/AnkitaRath/blob/main/aittapbl.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import cv2
import numpy as np
from tensorflow.keras.applications.mobilenet_v2 import MobileNetV2, preprocess_input, decode_predictions
from tensorflow.keras.preprocessing import image
from IPython.display import display, Javascript
from google.colab.output import eval_js
from base64 import b64decode

# Load the pre-trained MobileNetV2 model
model = MobileNetV2(weights='imagenet')

# Function to decode the image from the webcam
def take_photo(filename='img.jpg', quality=0.8):
    js = Javascript('''
    async function takePhoto(quality) {
        const div = document.createElement('div');
        const capture = document.createElement('button');
        capture.textContent = 'Capture';
        div.appendChild(capture);
        const video = document.createElement('video');
        const stream = await navigator.mediaDevices.getUserMedia({ 'video': true });
        document.body.appendChild(div);
        div.appendChild(video);
        video.srcObject = stream;
        await video.play();

        // Resize the output to fit the video element.
        google.colab.output.setIframeHeight(document.documentElement.scrollHeight, true);

        // Wait for the Capture to be clicked.
        await new Promise((resolve) => capture.onclick = resolve);
        const canvas = document.createElement('canvas');
        canvas.width = video.videoWidth;
        canvas.height = video.videoHeight;
        canvas.getContext('2d').drawImage(video, 0, 0);
        stream.getVideoTracks()[0].stop();
        div.remove();
        return canvas.toDataURL('image/jpg', quality);
    }
    ''')
    display(js)
    data = eval_js('takePhoto({})'.format(quality))
    binary = b64decode(data.split(',')[1])
    with open(filename, 'wb') as f:
        f.write(binary)
    return filename

# Function to predict the object in the image
def predict_object(filename):
    img = image.load_img(filename, target_size=(224, 224))
    img_array = image.img_to_array(img)
    img_array = np.expand_dims(img_array, axis=0)
    img_array = preprocess_input(img_array)
    predictions = model.predict(img_array)
    decoded_predictions = decode_predictions(predictions)
    # Get the top prediction
    prediction = decoded_predictions[0][0]
    object_name = prediction[1]
    confidence = prediction[2]
    return object_name, confidence

# Main function to capture image from webcam and predict the object
def main():
    while True:
        try:
            # Capture image from webcam
            filename = take_photo()
            print("Image captured!")
            # Predict object in the image
            object_name, confidence = predict_object(filename)
            print(f"Predicted object: {object_name} (Confidence: {confidence:.2f})")
        except Exception as e:
            print(f"Error: {e}")

# Run the main function
main()


<IPython.core.display.Javascript object>

Image captured!
1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step
Predicted object: orange (Confidence: 0.98)


<IPython.core.display.Javascript object>

Image captured!
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 100ms/step
Predicted object: goldfish (Confidence: 0.87)


<IPython.core.display.Javascript object>